In [1]:
import numpy
from sklearn.datasets import fetch_openml

# Fetch the dataset
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')

# The data is stored in the .data and .target attributes
X, y = fashion_mnist.data, fashion_mnist.target
# X is a NumPy array of shape (70000, 784) (flattened images)
# y is a NumPy array of shape (70000,) (labels as strings)

#split main df to make training, testing and validation sets
test_set = X[0:7000] #10% of df, 7k data points
valid_set = X[7000:14000] #10% of df, 7k data points
train_set = X[14000:] #80% of df, 56k data points



In [2]:
import torch

#conv to torch and move to cuda
test = torch.tensor(test_set, dtype=torch.float32).to('cuda')
valid = torch.tensor(valid_set, dtype=torch.float32).to('cuda')
train = torch.tensor(train_set, dtype=torch.float32).to('cuda')

y_test = torch.tensor(y_test, dtype=torch.long).to('cuda')
y_valid = torch.tensor(y_valid, dtype=torch.long).to('cuda')
y_train = torch.tensor(y_train, dtype=torch.long).to('cuda')

#nn class
class NeuralNetwork(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = torch.nn.Flatten()
    self.linear_relu_stack = torch.nn.Sequential(
      torch.nn.Linear(28*28,512),
      torch.nn.ReLU(),
      torch.nn.Linear(512,512),
      torch.nn.ReLU(),
      torch.nn.Linear(512,10)
    )

  def forward(self,x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

model = NeuralNetwork().to('cuda')
#criterion = torch.nn.MSELoss()
criterion = torch.nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#train
for epoch in range(1000):
    model.train()

    outputs = model(train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

logits = model(train)
pred_probab = torch.nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
numpy_pred = y_pred.cpu().detach().numpy()
y_train_numpy = y_train.cpu().detach().numpy()

d = numpy_pred == y_train_numpy
d.astype(int).sum()/len(d)